In [1]:
user_id = "tp99965073"
domain = "AXISB.COM"
app_name = "Fraud_&_AML"
yarn_queue = "root.DEV.AML_FRAUD_CS"

EXECUTOR_MEMORY=28
EXECUTOR_CORE=4

INITIAL_EXECUTORS=8
MAX_EXECUTORS=50     # 16x4 = 64 cores; lower this to your approved cap if smaller (YARN grants only what is free)
MIN_EXECUTORS=2

In [2]:
import sys
from pyspark.sql import SparkSession, SQLContext
from pyspark import SparkContext, SparkConf
from pyspark.sql import functions as F
from pyspark.sql import Window as W
import pyspark.sql.types as T
from datetime import datetime, date,timedelta
from dateutil.relativedelta import relativedelta
import pandas as pd
import time
import numpy as np


from IPython.core.display import HTML
display(HTML("<style>pre { white-space: pre !important; }</style>"))


# Construct the keytab path and the principal
keytab_path = f"/home/{user_id}@axisb.com/{user_id}.keytab"
principal = f"{user_id}@{domain}"

# Execute the commands
!kdestroy
!kinit -k -t "{keytab_path}" {principal}
try:
    spark.stop()
    print("Existing Spark Session Killed")
except:
    print("")
print("Creating new spark session since: ", datetime.now())
spark = SparkSession\
    .builder\
    .appName(f'JH_{user_id}_{app_name}')\
    .config('spark.sql.execution.arrow.enabled', "true")\
    .config('spark.master','yarn')\
    .config('spark.deploy-mode', 'cluster')\
    .config("spark.principal",f"{user_id}@AXISB.COM") \
    .config("spark.keytab",f"{user_id}.keytab")\
    .config('spark.dynamicAllocation.enabled','true') \
    .config('spark.dynamicAllocation.minExecutors',MIN_EXECUTORS) \
    .config('spark.dynamicAllocation.initialExecutors',INITIAL_EXECUTORS) \
    .config('spark.dynamicAllocation.maxExecutors',MAX_EXECUTORS) \
    .config('spark.executor.cores',EXECUTOR_CORE) \
    .config('spark.dynamicAllocation.executorAllocationRatio','0.8')\
    .config('spark.executor.memory',f'{EXECUTOR_MEMORY}g')\
    .config('spark.driver.memory','16g')\
    .config('spark.driver.cores','1')\
    .config('spark.executor.memoryOverhead','8g')\
    .config('spark.sql.shuffle.partitions','12000')\
    .config('spark.shuffle.service.enabled','true')\
    .config('spark.network.timeout','600s')\
    .config('spark.shuffle.io.maxRetries','10')\
    .config('spark.shuffle.io.retryWait','15s')\
    .config('spark.stage.maxConsecutiveAttempts','8')\
    .config('spark.maxRemoteBlockSizeFetchToMem','512m')\
    .config("spark.sql.broadcastTimeout", "-1")\
    .config("spark.sql.parquet.int96RebaseModeInRead", "LEGACY")\
    .config("spark.sql.legacy.timeParserPolicy", "LEGACY")\
    .config("spark.sql.parquet.int96RebaseModeInRead", "CORRECTED")\
    .config("spark.sql.parquet.int96RebaseModeInWrite", "CORRECTED")\
    .config("spark.sql.parquet.datetimeRebaseModeInRead", "CORRECTED")\
    .config("spark.sql.parquet.datetimeRebaseModeInWrite", "CORRECTED")\
    .config("spark.yarn.queue", yarn_queue)\
    .config('spark.sql.sources.partitionOverwriteMode', 'dynamic')\
    .config('spark.hadoop.hive.exec.dynamic.partition.mode', 'nonstrict')\
    .config('spark.sql.parquet.output.committer.class', 'org.apache.parquet.hadoop.ParquetOutputCommitter')\
    .config('spark.sql.sources.commitProtocolClass', 'org.apache.spark.sql.execution.datasources.SQLHadoopMapReduceCommitProtocol')\
    .config("spark.jars", "hdfs:///jars/iceberg-spark-runtime-3.3_2.12-1.6.1.jar,/opt/cloudera/parcels/CDH/jars/ojdbc8.jar")\
    .config('spark.sql.extensions', 'org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions')\
    .config('spark.sql.catalog.spark_catalog', 'org.apache.iceberg.spark.SparkSessionCatalog')\
    .config('spark.sql.catalog.spark_catalog.type', 'hive')\
    .config('spark.sql.adaptive.enabled', 'true')\
    .config('spark.sql.adaptive.coalescePartitions.enabled', 'true')\
    .config('spark.sql.adaptive.skewJoin.enabled', 'true')\
    .config('spark.sql.adaptive.autoBroadcastJoinThreshold','50MB')\
    .config("spark.sql.legacy.allowNonEmptyLocationInCTAS", "true")\
    .enableHiveSupport()\
    .getOrCreate()
sc = SparkContext.getOrCreate()
print("New Spark Session created at : ", datetime.now())

new_log_level = "ERROR"
sc.setLogLevel(new_log_level)
print(f"Logs level set to {new_log_level}")
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 200)


Creating new spark session since:  2026-06-25 04:07:08.521438


26/06/25 04:07:09 WARN  spark.SparkConf: [main]: The configuration key 'spark.maxRemoteBlockSizeFetchToMem' has been deprecated as of Spark 3.0 and may be removed in the future. Please use the new key 'spark.network.maxRemoteBlockSizeFetchToMem' instead.
26/06/25 04:07:09 WARN  spark.SparkConf: [main]: The configuration key 'spark.maxRemoteBlockSizeFetchToMem' has been deprecated as of Spark 3.0 and may be removed in the future. Please use the new key 'spark.network.maxRemoteBlockSizeFetchToMem' instead.
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/25 04:07:10 WARN  spark.SparkConf: [main]: The configuration key 'spark.maxRemoteBlockSizeFetchToMem' has been deprecated as of Spark 3.0 and may be removed in the future. Please use the new key 'spark.network.maxRemoteBlockSizeFetchToMem' instead.
26/06/25 04:07:10 WARN  spark.SparkConf: [Thread-10]: The configuration key 'spark.maxRemoteBlockSizeFetc

New Spark Session created at :  2026-06-25 04:07:21.988353
Logs level set to ERROR


# Notebook 02b (v2) — full transaction graph (all f_txn_sa)

Builds the v2 transaction-derived graph on **ALL** savings-account transactions (not the PL cohort),
tagging counterparties INTERNAL/EXTERNAL against the `axis_account_dim` from notebook 02.

Edges (mentor's schema `src, dst, edge_type, txn_ts, amount`):
* **TRANSFERRED_TO** — debit legs: `acc_no -> beneficiary_acc_no` (92.5% coverage; no txn_id pairing).
* **WITHDREW_AT** — ATM/POS legs: `acc_no -> atm_pos_id` (atm_pos_id = last token of `txn_remarks`).
* **SALARY_CREDIT** — salary-flagged credits: `salary_label_code -> acc_no`.
Nodes: ACCOUNT (INTERNAL/EXTERNAL + hub flag), ATM_POS. (CUSTOMER/SALARY_LABEL came from 02.)
Null `beneficiary_acc_no` and external inflows are kept as **node-flow features** (not fake nodes).

Window = Jun–Oct 2025 (union of the Sep/Oct/Nov trailing-3-month windows; leak-free per anchor in 03).
HEAVY (~3–4B debit legs) — run OFF-HOURS with a strong config. See `NEW_ARCH_SPEC_V2.md` / `Algorithm_Implementation_Plan.md`.

In [3]:
# ---- PARAMETERS ----
from pyspark.sql import functions as F
from pyspark import StorageLevel

WRITE_DB = "payment_cards_dev"; EMP = "TP99965073"
F_TXN_SA = "bdl_ods.f_txn_sa"
AXIS_DIM = f"{WRITE_DB}.{EMP}_v2_axis_account_dim"      # notebook 02 (INTERNAL keys + casa_type + salary_label)

# f_txn_sa columns
ACC, DST, AMT, DRCR = "acc_no", "beneficiary_acc_no", "txn_amt", "dr_cr"
PART = "tran_date"                                     # partition prune (string range)
CHANNEL, REMARKS, SALARY = "channel", "txn_remarks", "salary"
REVERSAL, SELF, UPI_APP = "reversal_date", "self_transfer_flag", "upi_app_name_beneficiary"

PULL_START, PULL_END = "2025-06-01", "2025-11-01"      # Jun–Oct (covers all anchor trailing windows)
HUB_FLOOR, HUB_CAP = 200, 10000                        # bounded data-driven fan-in hub cut

OUT_EDGES_TXN   = f"{WRITE_DB}.{EMP}_v2_edges_transferred"
OUT_EDGES_ATM   = f"{WRITE_DB}.{EMP}_v2_edges_withdrew"
OUT_EDGES_SALCR = f"{WRITE_DB}.{EMP}_v2_edges_salary_credit"
OUT_NODES_ACCT  = f"{WRITE_DB}.{EMP}_v2_nodes_account"
OUT_NODES_ATM   = f"{WRITE_DB}.{EMP}_v2_nodes_atm_pos"
OUT_NODE_FLOWS  = f"{WRITE_DB}.{EMP}_v2_node_flow_features"

def ACC_KEY(c): return F.upper(F.trim(F.col(c).cast("string")))
def save_overwrite(df, t):
    spark.sql("DROP TABLE IF EXISTS " + t); df.write.mode("overwrite").saveAsTable(t)

print("txn:", F_TXN_SA, "| window", PULL_START, "..", PULL_END, "| axis_dim:", AXIS_DIM)

txn: bdl_ods.f_txn_sa | window 2025-06-01 .. 2025-11-01 | axis_dim: payment_cards_dev.TP99965073_v2_axis_account_dim


## A. Confirm f_txn_sa schema (defensive)

In [4]:
print("f_txn_sa columns:", spark.table(F_TXN_SA).columns)

Hive Session ID = af94afb4-2861-49aa-aa26-035b57e33096


f_txn_sa columns: ['acid', 'acc_no', 'cust_id', 'acct_anchor_skey', 'cust_anchor_skey', 'scheme_code', 'scheme_type', 'sol_id_acc', 'sol_id_init', 'txn_id', 'part_tran_srl_num', 'txn_type', 'txn_sub_type', 'dr_cr', 'txn_amt', 'txn_particulars', 'txn_remarks', 'txn_remarks_2', 'ref_no', 'instr', 'instrmnt_date', 'instrmnt_num', 'txn_date', 'value_date', 'entry_date', 'pstd_date', 'vfd_date', 'lchg_time', 'rcre_time', 'entry_user_id', 'pstd_user_id', 'vfd_user_id', 'rcre_user_id', 'lchg_user_id', 'oth_txn_amt', 'tran_crncy_code', 'reversal_date', 'gl_sub_head_code', 'is_bi_txn', 'tagging', 'channel', 'transaction_type', 'transaction_subtype', 'purpose', 'atm_pos_id', 'location', 'salary', 'salary_logic', 'acct_name', 'beneficiary_name', 'beneficiary_bank', 'beneficiary_bank_cat', 'beneficiary_acc_no', 'beneficiary_phone_no', 'upi_app_name_axis', 'upi_app_name_beneficiary', 'self_transfer_flag', 'source_system_ref_no', 'tran_particular_eph', 'source', 'holiday', 'month_period', 'created_d

## B. Clean + derive (window, exclude reversals/self-transfers, positive amount)

In [5]:
_self = F.upper(F.trim(F.col(SELF).cast("string")))
ft = (spark.table(F_TXN_SA)
      .where((F.col(PART) >= F.lit(PULL_START)) & (F.col(PART) < F.lit(PULL_END)))
      .where(F.col(REVERSAL).isNull())                                   # exclude reversals
      .where(_self.isNull() | (~_self.isin("Y", "1", "YES", "TRUE")))    # exclude self-transfers (keep nulls)
      .where(F.col(AMT).cast("double") > 0)                              # positive amounts only
      .select(
          ACC_KEY(ACC).alias("acc"),
          ACC_KEY(DST).alias("benef"),
          F.col(AMT).cast("double").alias("amt"),
          (F.upper(F.trim(F.col(DRCR))) == "D").alias("is_debit"),
          (F.upper(F.trim(F.col(DRCR))) == "C").alias("is_credit"),
          F.date_format(F.to_date(F.col(PART)), "yyyyMM").alias("txn_month"),
          F.col(PART).cast("string").alias("txn_ts"),
          F.upper(F.trim(F.col(CHANNEL).cast("string"))).alias("chan"),
          F.col(REMARKS).cast("string").alias("remarks"),
          F.upper(F.trim(F.col(SALARY).cast("string"))).alias("sal_flag"),
          F.col(UPI_APP).cast("string").alias("upi_app")))
print("partition-pruned + cleaned f_txn_sa over", PULL_START, "..", PULL_END)

partition-pruned + cleaned f_txn_sa over 2025-06-01 .. 2025-11-01


## C. TRANSFERRED_TO edges (debit legs -> beneficiary account)

In [6]:
deb = ft.where(F.col("is_debit"))
# resolved transfers: beneficiary account present and not a self-loop
txn_ok = deb.where(F.col("benef").isNotNull() & (F.col("benef") != "") & (F.col("benef") != F.col("acc")))
edges_txn = (txn_ok.groupBy(F.col("acc").alias("src"), F.col("benef").alias("dst"), "txn_month")
             .agg(F.sum("amt").alias("sum_amount"), F.count("*").alias("txn_count"),
                  F.min("txn_ts").alias("first_ts"), F.max("txn_ts").alias("last_ts"))
             .withColumn("edge_type", F.lit("TRANSFERRED_TO")))
save_overwrite(edges_txn, OUT_EDGES_TXN)
edges_txn = spark.table(OUT_EDGES_TXN)
print("TRANSFERRED_TO edges:", edges_txn.count())

[Stage 4:======================================================>(284 + 2) / 286]

TRANSFERRED_TO edges: 1309051218


## D. WITHDREW_AT edges (ATM/POS legs -> atm_pos_id from txn_remarks)

In [7]:
_is_atmpos = F.col("chan").rlike("ATM|POS")
# exact SQL form proven in discovery D2 (avoids higher-order-function lambda edge cases)
_atm_id = F.expr("CASE WHEN remarks LIKE '%/%' "
                 "THEN trim(element_at(filter(split(remarks,'/'), x -> x IS NOT NULL AND x != ''), -1)) END")
atm = (ft.where(_is_atmpos).withColumn("atm_pos_id", _atm_id)
       .where(F.col("atm_pos_id").isNotNull() & (F.col("atm_pos_id") != "")))
edges_atm = (atm.groupBy(F.col("acc").alias("src"), F.col("atm_pos_id").alias("dst"), "txn_month")
             .agg(F.sum("amt").alias("sum_amount"), F.count("*").alias("txn_count"),
                  F.min("txn_ts").alias("first_ts"), F.max("txn_ts").alias("last_ts"))
             .withColumn("edge_type", F.lit("WITHDREW_AT")))
save_overwrite(edges_atm, OUT_EDGES_ATM)
edges_atm = spark.table(OUT_EDGES_ATM)
print("WITHDREW_AT edges:", edges_atm.count())

WITHDREW_AT edges: 42305036


## E. SALARY_CREDIT edges (salary-flagged credits -> salary_label_code -> account)

In [8]:
axis = spark.table(AXIS_DIM).select(F.col("acct").alias("acc"), "salary_label_code")
salcr = (ft.where(F.col("is_credit") & F.col("sal_flag").isin("Y", "1", "YES", "TRUE"))
         .join(axis, "acc", "inner")
         .where(F.col("salary_label_code").isNotNull() & (F.trim(F.col("salary_label_code")) != "")))
edges_salcr = (salcr.groupBy(
                  F.upper(F.trim(F.col("salary_label_code"))).alias("src"),
                  F.col("acc").alias("dst"), "txn_month")
               .agg(F.sum("amt").alias("sum_amount"), F.count("*").alias("txn_count"))
               .withColumn("edge_type", F.lit("SALARY_CREDIT")))
save_overwrite(edges_salcr, OUT_EDGES_SALCR)
edges_salcr = spark.table(OUT_EDGES_SALCR)
print("SALARY_CREDIT edges:", edges_salcr.count())

[Stage 18:=================================================>      (64 + 8) / 73]

SALARY_CREDIT edges: 22142118


## F. ACCOUNT nodes (tag INTERNAL/EXTERNAL via axis_dim) + data-driven hub flag

In [9]:
axis_keys = spark.table(AXIS_DIM).select(F.col("acct").alias("node_id"), "casa_type")

# all accounts appearing as a transfer endpoint
acct_ids = (edges_txn.select(F.col("src").alias("node_id"))
            .union(edges_txn.select(F.col("dst").alias("node_id")))
            .union(edges_atm.select(F.col("src").alias("node_id")))
            .distinct())

# tag via the Axis CASA universe
acct_nodes = (acct_ids.join(axis_keys, "node_id", "left")
              .withColumn("node_type", F.lit("ACCOUNT"))
              .withColumn("account_network_type",
                          F.when(F.col("casa_type").isNotNull(), F.lit("INTERNAL")).otherwise(F.lit("EXTERNAL"))))

# fan-in (distinct senders) per destination -> bounded data-driven hub cut
fanin = edges_txn.groupBy(F.col("dst").alias("node_id")).agg(F.countDistinct("src").alias("fan_in"))
_q = fanin.approxQuantile("fan_in", [0.999], 0.001)
HUB_CUT = int(min(max((_q[0] if _q and _q[0] else HUB_CAP), HUB_FLOOR), HUB_CAP))
print("fan-in p99.9 =", (_q[0] if _q else None), "-> bounded HUB_CUT =", HUB_CUT)

acct_nodes = (acct_nodes.join(fanin, "node_id", "left").fillna(0, ["fan_in"])
              .withColumn("is_hub", F.when(F.col("fan_in") > HUB_CUT, 1).otherwise(0)))
save_overwrite(acct_nodes, OUT_NODES_ACCT)
acct_nodes = spark.table(OUT_NODES_ACCT)
print("ACCOUNT nodes:", acct_nodes.count())
acct_nodes.groupBy("account_network_type").count().show()
print("hub accounts:", acct_nodes.where(F.col("is_hub") == 1).count())

fan-in p99.9 = 15450087.0 -> bounded HUB_CUT = 10000


ACCOUNT nodes: 224798839


+--------------------+---------+
|account_network_type|    count|
+--------------------+---------+
|            INTERNAL| 22017233|
|            EXTERNAL|202781606|
+--------------------+---------+

hub accounts: 759


## G. ATM_POS nodes

In [10]:
atm_nodes = (edges_atm.select(F.col("dst").alias("node_id")).distinct()
             .withColumn("node_type", F.lit("ATM_POS")))
save_overwrite(atm_nodes, OUT_NODES_ATM)
print("ATM_POS nodes:", spark.table(OUT_NODES_ATM).count())

[Stage 50:====================================================>   (71 + 4) / 76]

ATM_POS nodes: 603534


## H. Node-flow features (unresolved outflow + total inflow per account/month) — nothing dropped

In [11]:
unresolved_out = (deb.where(F.col("benef").isNull() | (F.col("benef") == ""))
                  .groupBy(F.col("acc").alias("node_id"), "txn_month")
                  .agg(F.sum("amt").alias("unresolved_outflow_amt"), F.count("*").alias("unresolved_outflow_cnt")))
inflow = (ft.where(F.col("is_credit"))
          .groupBy(F.col("acc").alias("node_id"), "txn_month")
          .agg(F.sum("amt").alias("inflow_amt"), F.count("*").alias("inflow_cnt")))
node_flows = unresolved_out.join(inflow, ["node_id", "txn_month"], "outer").fillna(0)
save_overwrite(node_flows, OUT_NODE_FLOWS)
print("node_flow_features rows:", spark.table(OUT_NODE_FLOWS).count())

node_flow_features rows: 119994274


## Done — QC summary

In [12]:
print("=== Notebook 02b (v2) outputs ===")
for t in [OUT_EDGES_TXN, OUT_EDGES_ATM, OUT_EDGES_SALCR, OUT_NODES_ACCT, OUT_NODES_ATM, OUT_NODE_FLOWS]:
    print("  %-58s rows: %d" % (t, spark.table(t).count()))
print("TRANSFERRED_TO by month:")
edges_txn.groupBy("txn_month").count().orderBy("txn_month").show()
print("NOTEBOOK 02b (v2) COMPLETE — full transaction graph (TRANSFERRED_TO + WITHDREW_AT + SALARY_CREDIT),")
print("ACCOUNT nodes tagged INTERNAL/EXTERNAL + hub flag, ATM_POS nodes, node-flow features.")
print("Paste back: edge/node counts, INTERNAL/EXTERNAL split, hub accounts, TRANSFERRED_TO-by-month — then 03 (v2).")

=== Notebook 02b (v2) outputs ===


  payment_cards_dev.TP99965073_v2_edges_transferred          rows: 1309051218


  payment_cards_dev.TP99965073_v2_edges_withdrew             rows: 42305036
  payment_cards_dev.TP99965073_v2_edges_salary_credit        rows: 22142118
  payment_cards_dev.TP99965073_v2_nodes_account              rows: 224798839
  payment_cards_dev.TP99965073_v2_nodes_atm_pos              rows: 603534


  payment_cards_dev.TP99965073_v2_node_flow_features         rows: 119994274
TRANSFERRED_TO by month:


[Stage 79:====================================================> (279 + 7) / 286]

+---------+---------+
|txn_month|    count|
+---------+---------+
|   202506|244431263|
|   202507|256677507|
|   202508|266961545|
|   202509|262121871|
|   202510|278859032|
+---------+---------+

NOTEBOOK 02b (v2) COMPLETE — full transaction graph (TRANSFERRED_TO + WITHDREW_AT + SALARY_CREDIT),
ACCOUNT nodes tagged INTERNAL/EXTERNAL + hub flag, ATM_POS nodes, node-flow features.
Paste back: edge/node counts, INTERNAL/EXTERNAL split, hub accounts, TRANSFERRED_TO-by-month — then 03 (v2).
